## tl;dr

- **Overall assessment: Needs revision.** The order-review export reconciles cleanly, but the stock-gap shortage definition and SKU-concentration window are not stable enough for decision use.
- **Order review passed its arithmetic backtest:** 2,571 rows, no duplicate product keys, and exact agreement between hidden calculations, displayed values, and summary totals.
- **Stock-gap shortage is not reproducible from the screen definition for 30 of 75 gap rows.** Those rows show 319,995 units versus 149,065 units from displayed gap days × displayed sales speed, a +170,930 unit difference.
- **Historical backtests are not deterministic** because one no-ETA status branch references the execution date instead of the analysis base date.
- **SKU concentration includes out-of-window sales rows** and can render missing identity values as the literal text `nan`.


## Context & Methods

This diagnostic notebook validates the full order-analysis area: order review, stock gap, SKU concentration, shared summaries, and Excel exports. It uses aggregate-only evidence from the latest available local exports and synthetic boundary cases; no product-level identifiers are displayed.

### Key Assumptions

- The latest matching local export is representative of the current screen implementation.
- `발주필요수량`, `권장 발주 금액`, scenario stock, and MOI must reconcile to the hidden calculation sheet.
- The stock-gap help text defines shortage as `gap days × daily sales speed`; one-decimal display rounding is allowed in the tolerance band.
- Historical status decisions should be a pure function of the supplied analysis base date.


## Data

In [1]:
import json
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
export_results = json.loads((HERE / "results.json").read_text(encoding="utf-8"))
logic_results = json.loads((HERE / "logic_results.json").read_text(encoding="utf-8"))
export_results["as_of"]


'2026-07-15T11:58:55.655393+09:00'

## Results

In [2]:
order = export_results["order_export"]
stock = export_results["stock_gap_export"]

summary = pd.DataFrame([
    {"area": "Order review", "population": order["row_count"], "check": "Scenario formulas / summaries", "result": "PASS" if all([order["scenario_stock_failures"] == 0, order["scenario_qty_failures"] == 0, order["scenario_amount_failures"] == 0, order["moi_failures"] == 0, order["summary_qty_matches"], order["summary_amount_matches"]]) else "FAIL"},
    {"area": "Stock gap", "population": stock["row_count"], "check": "Summary cards", "result": "PASS" if all(stock["summary_matches"].values()) else "FAIL"},
    {"area": "Stock gap", "population": stock["computed_risk"], "check": "Shortage formula reproducibility", "result": f'FAIL ({stock["shortage_reconciliation_exceptions"]} rows)'},
    {"area": "SKU concentration", "population": 2, "check": "Three-month date window boundary", "result": "FAIL" if logic_results["sku_concentration_window"]["out_of_window_row_included"] else "PASS"},
])
summary


,area,population,check,result
0,Order review,2571,Scenario formulas / summaries,PASS
1,Stock gap,2570,Summary cards,PASS
2,Stock gap,75,Shortage formula reproducibility,FAIL (30 rows)
3,SKU concentration,2,Three-month date window boundary,FAIL


In [3]:
pd.DataFrame([
    {"metric": "Order rows", "observed": order["row_count"], "expected": order["target_count"], "status": order["target_count_matches"]},
    {"metric": "Before-scenario order qty", "observed": order["before_qty_sum"], "expected": order["summary_qty"], "status": order["summary_qty_matches"]},
    {"metric": "Before-scenario order amount (KRW)", "observed": order["before_amount_sum_krw"], "expected": order["summary_amount_krw"], "status": order["summary_amount_matches"]},
    {"metric": "Duplicate product-code rows", "observed": order["duplicate_product_code_rows"], "expected": 0, "status": order["duplicate_product_code_rows"] == 0},
    {"metric": "Formula/cache mismatches", "observed": order["visible_cache_failures"] + order["formula_pattern_failures"], "expected": 0, "status": order["visible_cache_failures"] + order["formula_pattern_failures"] == 0},
])


,metric,observed,expected,status
0,Order rows,2.571000e+03,2.571000e+03,True
1,Before-scenario order qty,1.738702e+06,1.738702e+06,True
2,Before-scenario order amount (KRW),9.729023e+09,9.729023e+09,True
3,Duplicate product-code rows,0.000000e+00,0.000000e+00,True
4,Formula/cache mismatches,0.000000e+00,0.000000e+00,True


In [4]:
pd.DataFrame([
    {"metric": "Gap-risk rows", "displayed": stock["summary_risk"], "recomputed": stock["computed_risk"], "difference": stock["summary_risk"] - stock["computed_risk"]},
    {"metric": "7+ day gap rows", "displayed": stock["summary_long_gap"], "recomputed": stock["computed_long_gap"], "difference": stock["summary_long_gap"] - stock["computed_long_gap"]},
    {"metric": "Shortage units on 30 exceptions", "displayed": stock["shortage_exception_displayed_total"], "recomputed": stock["shortage_exception_visible_formula_total"], "difference": stock["shortage_exception_delta"]},
    {"metric": "Year-ambiguous date rows", "displayed": stock["gap_date_ambiguous_rows"], "recomputed": None, "difference": None},
])


,metric,displayed,recomputed,difference
0,Gap-risk rows,75,75.0,0.0
1,7+ day gap rows,53,53.0,0.0
2,Shortage units on 30 exceptions,319995,149065.0,170930.0
3,Year-ambiguous date rows,70,NaN,NaN


In [5]:
pd.DataFrame([
    {"case": "Historical no-ETA status", **logic_results["stock_gap_historical_base_date"]},
    {"case": "SKU three-month window", **logic_results["sku_concentration_window"]},
    {"case": "Missing SKU identity", **logic_results["sku_concentration_missing_identity"]},
])


,case,baseDate,stockoutDate,observedStatus,expectedStatusUsingBaseDate,declared_window,expected_in_window_qty,expected_in_window_amount_krw,observed_qty,observed_amount_krw,out_of_window_row_included,observed_product_name,observed_brand,nan_rendered_as_identity
0,Historical no-ETA status,2026-01-01,2026-04-10,needs_check,sufficient,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SKU three-month window,NaN,NaN,NaN,NaN,2026-02-01 through 2026-04-30,1.0,100.0,10.0,1000.0,True,NaN,NaN,NaN
2,Missing SKU identity,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan,nan,True


In [6]:
# Reconciliation assertions that are expected to pass.
assert order["row_count"] == order["target_count"]
assert order["duplicate_product_code_rows"] == 0
assert order["scenario_stock_failures"] == 0
assert order["scenario_qty_failures"] == 0
assert order["scenario_amount_failures"] == 0
assert order["moi_failures"] == 0
assert order["summary_qty_matches"] and order["summary_amount_matches"]
assert all(stock["summary_matches"].values())
assert stock["gap_date_failures"] == 0
print("All expected-pass reconciliation assertions succeeded.")


All expected-pass reconciliation assertions succeeded.


## Takeaways

1. **Keep the order-review arithmetic.** Its row grain, scenario formulas, cached values, and totals are internally consistent on the reviewed export.
2. **Choose one canonical stock-gap shortage definition.** Either display the explicit order-model shortage and rename/explain it, or calculate the displayed shortage strictly from gap days × daily sales speed. Do not mix the two silently.
3. **Make status calculations base-date pure.** Pass the analysis base date into the no-ETA branch instead of reading the machine clock.
4. **Filter SKU concentration by the requested analysis window before grouping.** Record the applied start/end dates in result metadata.
5. **Normalize missing identity fields.** Treat null, NaN, blank, `none`, and `-` as missing and fall back to the product master.
6. **Use full dates in stock-gap exports.** Seventy rows cannot be temporally interpreted from `MM/DD` alone because the year is omitted.
7. **Fix Excel validation XML.** Replace the non-standard validation error style with a standards-compliant value so external parsers can open the workbook.
